# Notebook 06 — RLHF Extension

**Steps:**
1. Build preference dataset from human eval + DeepSeek-as-judge
2. Train reward model (1 epoch, LoRA r=8)
3. PPO training (2 epochs)
4. Compare PPO model vs Config D on sample questions

> **Prerequisites:** Notebooks 01–05 must be complete.  
> **Note:** With ~80-100 preference pairs, PPO will not produce dramatic improvements.
> Document the pipeline and KL divergence behavior — the learning objective here is demonstrating RLHF, not large score gains.

In [ ]:
from pathlib import Path
import os

# Tự động tìm đường dẫn
if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NLP_Final/Source/')
else:
    BASE = Path("../")

print(f'Base directory: {BASE}')
print(os.listdir(BASE))

In [ ]:
%%capture
!pip install -q \
    transformers==4.45.0 peft==0.13.0 trl==0.11.4 \
    accelerate==1.0.1 datasets==3.1.0 \
    openai
!pip install -q -U bitsandbytes
# !pip install -q "numpy<2" faiss-cpu==1.8.0 sentence-transformers==3.2.1
print('Packages installed.')

## 1. Load Human Evaluation Scores

In [ ]:
import csv, json, os, random, time
import torch

RESULTS_DIR    = f'{BASE}/results'
PREF_PATH      = f'{RESULTS_DIR}/preference_dataset.jsonl'
DEEPSEEK_CACHE = f'{RESULTS_DIR}/deepseek_judged.jsonl'

SCORE_COLS = ['accuracy_1_5', 'completeness_1_5', 'fluency_1_5',
              'conciseness_1_5', 'no_hallucination_1_5']

SYSTEM_PROMPT = (
    'Bạn là trợ lý AI của Trường Đại học Tôn Đức Thắng (TDTU). '
    'Bạn trả lời các câu hỏi về quy chế, quy định, chính sách của trường một cách chính xác và đầy đủ. '
    'Trả lời bằng tiếng Việt. Nếu không có đủ thông tin, hãy nói rõ điều đó.'
)

def total_score(row: dict) -> float:
    try:
        return sum(float(row[k]) for k in SCORE_COLS if row.get(k, '').strip())
    except (ValueError, TypeError):
        return 0.0

def fmt_chatml(question: str, chosen: str, rejected: str) -> dict:
    ctx = (f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
           f'<|im_start|>user\n{question}<|im_end|>\n')
    return {
        'chosen':   ctx + f'<|im_start|>assistant\n{chosen}<|im_end|>',
        'rejected': ctx + f'<|im_start|>assistant\n{rejected}<|im_end|>',
    }

# Load human eval CSV
human_scores = []
with open(f'{RESULTS_DIR}/human_eval_template.csv', 'r', encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        if total_score(row) > 0:
            human_scores.append(row)

# Load per-config predictions (saved by NB05)
cfg_preds = {}
for cfg in ['A', 'B', 'C', 'D']:
    rows = []
    with open(f'{RESULTS_DIR}/config_{cfg}_results.jsonl', 'r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line.strip()))
    cfg_preds[cfg] = rows

print(f'Human eval rows : {len(human_scores)}')
print(f'Test predictions: {len(cfg_preds["A"])} questions × 4 configs')

Human eval rows : 400
Test predictions: 284 questions × 4 configs




## 2. Build Preference Dataset from Human Scores

In [ ]:
from collections import defaultdict
from itertools import combinations

by_question = defaultdict(list)
for row in human_scores:
    by_question[row['question_id']].append(row)

preference_pairs = []
for qid, rows in by_question.items():
    for r1, r2 in combinations(rows, 2):
        s1, s2 = total_score(r1), total_score(r2)
        if s1 == s2:
            continue
        chosen   = r1 if s1 > s2 else r2
        rejected = r2 if s1 > s2 else r1
        pair = fmt_chatml(
            chosen['question'], chosen['prediction'], rejected['prediction']
        )
        pair['source'] = "human"
        preference_pairs.append(pair)
print(f'Preference pairs from human eval: {len(preference_pairs)}')

Preference pairs from human eval: 510


## 3. Augment with Deepseek-as-Judge

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv(f'{BASE}/.env')

DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
deepseek_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url='https://api.deepseek.com',
)

JUDGE_PROMPT = """Bạn là chuyên gia đánh giá câu trả lời về quy chế đại học TDTU.

Câu hỏi: {question}
Tài liệu tham khảo: {reference}

Câu trả lời A:
{answer_A}

Câu trả lời B:
{answer_B}

Câu trả lời nào chính xác hơn, đầy đủ hơn và tự nhiên hơn so với tài liệu tham khảo?
Trả về JSON: {{"winner": "A" hoặc "B", "reason": "lý do ngắn"}}"""

def deepseek_judge(question, reference, answer_a, answer_b):
    prompt = JUDGE_PROMPT.format(
        question=question, reference=reference,
        answer_A=answer_a[:600], answer_B=answer_b[:600],
    )
    try:
        resp = deepseek_client.chat.completions.create(
            model='deepseek-v4-flash',
            messages=[{'role': 'user', 'content': prompt}],
            response_format={'type': 'json_object'},
            temperature=0.1,
        )
        return json.loads(resp.choices[0].message.content).get('winner')
    except Exception as e:
        print(f'  Judge error: {e}')
        return None

print('DeepSeek judge ready.')

DeepSeek judge ready.


In [ ]:
from tqdm import tqdm

# Resume: load các pairs DeepSeek đã judge từ lần chạy trước
extra_pairs  = []
judged_keys  = set()
if os.path.exists(DEEPSEEK_CACHE):
    with open(DEEPSEEK_CACHE, 'r', encoding='utf-8') as f:
        for line in f:
            p = json.loads(line.strip())
            extra_pairs.append(p)
            judged_keys.add(p['_key'])
    print(f'Resumed: {len(extra_pairs)} DeepSeek pairs from cache')

human_qids = set()
with open(f'{RESULTS_DIR}/human_eval_template.csv', 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for row in reader:
        human_qids.add(row['question_id'])

# Lọc indices chưa dùng
remaining = [i for i in range(len(cfg_preds['A']))
             if cfg_preds['A'][i].get('id', f'q{i}') not in human_qids]

ROUNDS = [('D', 'A', remaining), ('C', 'A', remaining)]

TARGET = 1000

# BATCH PROCESSING
batch_buffer = []
BATCH_SIZE = 20
with open(DEEPSEEK_CACHE, 'a', encoding='utf-8') as f:
    for cfg_w, cfg_l, indices in ROUNDS:
        round_name = f'{cfg_w}_vs_{cfg_l}'
        print(f'\n--- {round_name}: {len(indices)} questions ---')

        # Progress bar cho round hien tai
        for idx in tqdm(indices, desc=round_name, unit='q'):
            key = f'{round_name}_{idx}'
            if key in judged_keys:
                continue
            if len(preference_pairs) + len(extra_pairs) >= TARGET:
                print(f'Reached {TARGET} — stopping.')
                break

            item_w = cfg_preds[cfg_w][idx]
            item_l = cfg_preds[cfg_l][idx]
            winner = deepseek_judge(item_w['question'], item_w['reference'],
                                    item_w['prediction'], item_l['prediction'])

            if winner == 'A':
                chosen, rejected = item_w['prediction'], item_l['prediction']
            elif winner == 'B':
                chosen, rejected = item_l['prediction'], item_w['prediction']
            else:
                time.sleep(0.5)
                continue

            pair = fmt_chatml(item_w['question'], chosen, rejected)
            pair['_key'] = key
            pair['source'] = "llm_judge"

            batch_buffer.append(pair)
            judged_keys.add(key)
            extra_pairs.append(pair)

            if len(batch_buffer) >= BATCH_SIZE:
                for p in batch_buffer:
                    f.write(json.dumps(p, ensure_ascii=False) + '\n')
                f.flush()
                batch_buffer.clear()
                time.sleep(0.5)
            tqdm.write(f'Collected: {len(preference_pairs) + len(extra_pairs)}/{TARGET} pairs')

        if batch_buffer:
            for p in batch_buffer:
                f.write(json.dumps(p, ensure_ascii=False) + '\n')
            f.flush()
            batch_buffer.clear()

        print(f'Total so far: {len(preference_pairs) + len(extra_pairs)} pairs')
        if len(preference_pairs) + len(extra_pairs) >= TARGET:
            break

print(f'\nDeepSeek pairs: {len(extra_pairs)}')
print(f'Total pairs: {len(preference_pairs) + len(extra_pairs)}')

Resumed: 368 DeepSeek pairs from cache

--- D_vs_A: 184 questions ---


D_vs_A: 100%|██████████| 184/184 [00:00<00:00, 575075.96q/s]


Total so far: 878 pairs

--- C_vs_A: 184 questions ---


C_vs_A: 100%|██████████| 184/184 [00:00<00:00, 689680.01q/s]

Total so far: 878 pairs

DeepSeek pairs: 368
Total pairs: 878


In [ ]:
import random
import json

all_pairs = preference_pairs + extra_pairs
print(f'Human eval pairs : {len(preference_pairs)}')
print(f'llm pairs     : {len(extra_pairs)}')
print(f'Total            : {len(all_pairs)}')

human_pairs = preference_pairs.copy()
llm_pairs = extra_pairs.copy()

random.seed(42)
random.shuffle(human_pairs)
random.shuffle(llm_pairs)

split_human = int(len(human_pairs) * 0.9)
split_llm = int(len(llm_pairs) * 0.9)

train_pairs = human_pairs[:split_human] + llm_pairs[:split_llm]
val_pairs = human_pairs[split_human:] + llm_pairs[split_llm:]

random.shuffle(train_pairs)
random.shuffle(val_pairs)

print(f"Train: {len(train_pairs)} (human: {split_human}, llm: {split_llm})")
print(f"Val  : {len(val_pairs)} (human: {len(human_pairs)-split_human}, llm: {len(llm_pairs)-split_llm})")

# lưu full dataset
with open(PREF_PATH, 'w', encoding='utf-8') as f:
    for p in all_pairs:
        f.write(json.dumps({'chosen': p['chosen'], 'rejected': p['rejected'], 'source': p['source']},
                           ensure_ascii=False) + '\n')
print(f'Saved → {PREF_PATH}')

# Lưu train pairs
with open(f'{RESULTS_DIR}/train_pairs.jsonl', 'w', encoding='utf-8') as f:
    for p in train_pairs:
        f.write(json.dumps({
            'chosen': p['chosen'],
            'rejected': p['rejected'],
            'source': p['source']
        }, ensure_ascii=False) + '\n')

# Lưu val pairs
with open(f'{RESULTS_DIR}/val_pairs.jsonl', 'w', encoding='utf-8') as f:
    for p in val_pairs:
        f.write(json.dumps({
            'chosen': p['chosen'],
            'rejected': p['rejected'],
            'source': p['source']
        }, ensure_ascii=False) + '\n')
print(f'Saved train/val pairs → {RESULTS_DIR}')

Human eval pairs : 510
llm pairs     : 368
Total            : 878
Train: 790 (human: 459, llm: 331)
Val  : 88 (human: 51, llm: 37)
Saved → /content/drive/MyDrive/NLP_Final/Source/results/preference_dataset.jsonl
Saved train/val pairs → /content/drive/MyDrive/NLP_Final/Source/results


## 4. Train Reward Model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import RewardTrainer, RewardConfig
import torch
from datasets import load_dataset
# T4 optimization
if torch.cuda.is_available():
    # T4 có 16GB VRAM, dùng bfloat16 là tốt nhất
    COMPUTE_DTYPE = torch.bfloat16
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    COMPUTE_DTYPE = torch.float32

MODEL_ID    = 'Qwen/Qwen2.5-3B-Instruct'
REWARD_PATH = f'{BASE}/models/reward_model'
train_dataset = load_dataset('json', data_files=f'{RESULTS_DIR}/train_pairs.jsonl', split='train')
val_dataset = load_dataset('json', data_files=f'{RESULTS_DIR}/val_pairs.jsonl', split='train')


reward_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if reward_tokenizer.pad_token is None:
    reward_tokenizer.pad_token = reward_tokenizer.eos_token

def tokenize_reward(example):
    # Tokenize với padding='max_length' để đảm bảo cùng độ dài
    tok_c = reward_tokenizer(
        example['chosen'],
        truncation=True,
        max_length=512,
        padding='max_length'
    )

    tok_r = reward_tokenizer(
        example['rejected'],
        truncation=True,
        max_length=512,
        padding='max_length'
    )

    return {
        'input_ids_chosen': tok_c['input_ids'],
        'attention_mask_chosen': tok_c['attention_mask'],
        'input_ids_rejected': tok_r['input_ids'],
        'attention_mask_rejected': tok_r['attention_mask'],
    }

# Tokenize dataset
print("Tokenizing dataset with padding...")
tokenized_train = train_dataset.map(
    tokenize_reward,
    batched=False,
    remove_columns=['chosen', 'rejected'],
    desc='Tokenizing train',
)
tokenized_train.set_format('torch')

tokenized_val = val_dataset.map(
    tokenize_reward,
    batched=False,
    remove_columns=['chosen', 'rejected'],
    desc='Tokenizing val',
)
tokenized_val.set_format('torch')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

reward_model_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=1,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    use_cache=False,  # Cần cho gradient checkpointing
    attn_implementation="sdpa",  # SDPA nhanh hơn trên T4
)

reward_model_base.config.pad_token_id = reward_tokenizer.pad_token_id
reward_model_base = prepare_model_for_kbit_training(reward_model_base)

reward_lora = LoraConfig(
    r=4, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='SEQ_CLS',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
reward_model = get_peft_model(reward_model_base, reward_lora)
reward_model.print_trainable_parameters()

GPU: Tesla T4
VRAM: 15.6 GB


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-3B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,845,248 || all params: 3,087,785,984 || trainable%: 0.0598


In [ ]:
import warnings
warnings.filterwarnings('ignore')

reward_config = RewardConfig(
    output_dir=REWARD_PATH,
    num_train_epochs=1,
    per_device_train_batch_size=2,        # Batch nhỏ cho T4 16GB
    per_device_eval_batch_size=2,         # Batch nhỏ cho eval
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=10,
    eval_steps=50,                       # Đánh giá mỗi 50 steps
    save_steps=500,
    save_total_limit=2,
    max_length=512,

    bf16=COMPUTE_DTYPE == torch.bfloat16,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    report_to='none',
    evaluation_strategy='steps',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
)

# Kiểm tra memory trước khi train
print(f"\nMemory before training: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

reward_trainer = RewardTrainer(
    model=reward_model,
    tokenizer=reward_tokenizer,
    args=reward_config,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

print('Training reward model...')
reward_trainer.train()
reward_trainer.save_model(REWARD_PATH)
reward_tokenizer.save_pretrained(REWARD_PATH)
print(f'Reward model saved → {REWARD_PATH}')

# Đánh giá cuối cùng
print("\nFinal evaluation:")
eval_results = reward_trainer.evaluate()
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

print("\n-- Reward Model training completed!")

In [ ]:
reward_trainer.save_model(REWARD_PATH)
reward_tokenizer.save_pretrained(REWARD_PATH)
print(f'Reward model saved → {REWARD_PATH}')

# Đánh giá cuối cùng
print("\nFinal evaluation:")
eval_results = reward_trainer.evaluate()
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

print("\n-- Reward Model training completed!")

Reward model saved → /content/drive/MyDrive/NLP_Final/Source/models/reward_model

Final evaluation:


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ <|im_start|>system                            │ <|im_start|>system                           │ [0.8284, 0.1716] │
│ Bạn là trợ lý AI của Trường Đại học Tôn Đức   │ Bạn là trợ lý AI của Trường Đại học Tôn Đức  │                  │
│ Thắng (TDTU). Bạn trả lời các câu hỏi về quy  │ Thắng (TDTU). Bạn trả lời các câu hỏi về quy │                  │
│ chế, quy định, chính sách của trường một cách │ chế, quy định, chính sách của trường một     │                  │
│ chính xác và đầy đủ. Trả lời bằng tiếng Việt. │ cách chính xác và đầy đủ. Trả lời bằng tiếng │                  │
│ Nếu không có đủ thông tin, hãy nói rõ điều    │ Việt. Nếu không có đủ thông tin, hãy nói rõ  │                  │
│ đó.<|im_end|>                                 │ điều đó.<|im_end|>                           │                  │
│ <|im_start|>user                              │ <|im_start|>user                             │                  │
│ Em làm tổ trưởng thì được tính điểm rèn luyện │ Em làm tổ trưởng thì được tính điểm rèn      │                  │
│ thế nào ạ?<|im_end|>                          │ luyện thế nào ạ?<|im_end|>                   │                  │
│ <|im_start|>assistant                         │ <|im_start|>assistant                        │                  │
│ Để biết chính xác quy định về việc tính điểm  │ Theo các thông tin mà bạn cung cấp, để được  │                  │
│ rèn luyện cho vị trí tổ trưởng của bạn tại    │ tính điểm rèn luyện, sinh viên tổ trưởng cần │                  │
│ Trường Đại học Tôn Đức Thắng, tôi khuyên bạn  │ đảm bảo rằng kết quả đánh giá rèn luyện học  │                  │
│ nên tham khảo trực tiếp từ Ban Giám hiệu hoặc │ kỳ gần nhất đạt từ 65 điểm trở lên. Điều này │                  │
│ liên hệ với phòng Công tác sinh viên của      │ có nghĩa là điểm rèn luyện của bạn phải đạt  │                  │
│ trường. Thông thường, các trường đại học sẽ   │ từ 65 điểm trở lên mới đủ điều kiện để được  │                  │
│ có quy chế riêng về việc tính điểm rèn luyện  │ tính điểm.                                   │                  │
│ cho từng vị trí công tác, bao gồm cả vai trò  │                                              │                  │
│ tổ trưởng. Nếu trường không có quy định cụ    │ Tuy nhiên, điểm rèn luyện chỉ là một phần    │                  │
│ thể, Ban Giám hiệu sẽ là nguồn thông tin      │ của việc đánh giá. Để được xem xét cho điểm  │                  │
│ chính xác nhất.<|im_end|>                     │ rèn luyện, bạn cũng cần tuân thủ các yêu cầu │                  │
│                                               │ khác như:                                    │                  │
│                                               │                                              │                  │
│                                               │ 1. Gương mẫu trong học tập và rèn            │                  │
│                                               │ luyện<|im_end|>                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ <|im_start|>system                            │ <|im_start|>system                           │ [0.8256, 0.1744] │
│ Bạn là trợ lý AI của Trường Đại học Tôn Đức   │ Bạn là trợ lý AI của Trường Đại học Tôn Đức  │                  │
│ Thắng (TDTU). Bạn trả lời các câu hỏi về quy  │ Thắng (TDTU). Bạn trả lời các câu hỏi về quy │                  │
│ chế, quy định, chính sách của trường một cách │ chế, quy định, chính sách của trường một     │                  │
│ chính xác và đầy đủ. Trả lời bằng tiếng Việt. │ cách c

Step,Training Loss,Validation Loss,Accuracy
50,0.650000,0.557099,0.715909
72,0.630000,0.548032,0.727273


  eval_loss: 0.5480
  eval_accuracy: 0.7273

-- Reward Model training completed!


## 5. PPO Training

In [ ]:
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from peft import PeftModel
from transformers import AutoTokenizer

SFT_PATH  = f'{BASE}/models/sft_checkpoint'
PPO_PATH  = f'{BASE}/models/ppo_checkpoint'

ppo_tokenizer = AutoTokenizer.from_pretrained(SFT_PATH, trust_remote_code=True)
if ppo_tokenizer.pad_token is None:
    ppo_tokenizer.pad_token = ppo_tokenizer.eos_token
ppo_tokenizer.padding_side = 'left'  # PPO requires left-padding

# Policy model (initialised from SFT checkpoint)
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    SFT_PATH,
    torch_dtype=COMPUTE_DTYPE,
    device_map='auto',
    trust_remote_code=True,
)

# Reference model (frozen copy of SFT checkpoint for KL penalty)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    SFT_PATH,
    torch_dtype=COMPUTE_DTYPE,
    device_map='auto',
    trust_remote_code=True,
)

print('PPO models loaded.')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PPO models loaded.


In [ ]:
ppo_config = PPOConfig(
    learning_rate=1e-5,
    batch_size=4,
    mini_batch_size=2,
    ppo_epochs=2,
    kl_penalty='kl',
    init_kl_coef=0.2,
    adap_kl_ctrl=True,
    target_kl=6.0,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=ref_model,
    tokenizer=ppo_tokenizer,
)

print('PPO trainer initialized.')

PPO trainer initialized.


In [ ]:
from tqdm import tqdm

# Load training questions for PPO
train_pairs = []
with open(f'{BASE}/data/qa_filtered/qa_train.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        train_pairs.append(json.loads(line.strip()))

SYSTEM_PROMPT = (
    'Bạn là trợ lý AI của Trường Đại học Tôn Đức Thắng (TDTU). '
    'Trả lời bằng tiếng Việt, chính xác và đầy đủ.'
)

def score_with_reward_model(text: str) -> float:
    inputs = reward_tokenizer(text, return_tensors='pt', truncation=True,
                               max_length=512).to(reward_model.device)
    with torch.no_grad():
        output = reward_model(**inputs)
    return output.logits.squeeze().item()


print('Starting PPO training...')
ppo_stats = []

random.shuffle(train_pairs)
ppo_pairs = train_pairs[:min(50, len(train_pairs))]  # Use 50 samples for PPO

for epoch in range(2):
    print(f'PPO Epoch {epoch+1}/2')
    for i in range(0, len(ppo_pairs), ppo_config.batch_size):
        batch = ppo_pairs[i:i + ppo_config.batch_size]
        if not batch:
            break

        # Tokenize queries
        query_texts = [
            f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
            f'<|im_start|>user\n{p["question"]}<|im_end|>\n'
            f'<|im_start|>assistant\n'
            for p in batch
        ]
        query_tensors = [
            ppo_tokenizer.encode(q, return_tensors='pt').squeeze()
            for q in query_texts
        ]

        # Generate responses
        response_tensors = ppo_trainer.generate(
            query_tensors, max_new_tokens=256, do_sample=True, temperature=0.8,
        )

        # Decode and score
        responses = [ppo_tokenizer.decode(r, skip_special_tokens=True) for r in response_tensors]
        rewards   = [torch.tensor(score_with_reward_model(r)) for r in responses]

        # PPO step
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        ppo_stats.append(stats)

        kl = stats.get('objective/kl', 0)
        print(f'  Batch {i//ppo_config.batch_size}: reward_mean={sum(r.item() for r in rewards)/len(rewards):.3f}, KL={kl:.3f}')

        if kl > 20:
            print('⚠️  KL divergence too high — stopping PPO early.')
            break

print('\nPPO training complete.')

In [ ]:
# Save PPO model
os.makedirs(PPO_PATH, exist_ok=True)
ppo_model.save_pretrained(PPO_PATH)
ppo_tokenizer.save_pretrained(PPO_PATH)
print(f'PPO model saved → {PPO_PATH}')

PPO model saved → /content/drive/MyDrive/NLP_Final/Source/models/ppo_checkpoint


## 6. Compare PPO Model vs Config D on Sample Questions

In [ ]:
# Load test pairs and Config D predictions for comparison
test_pairs = []
with open(f'{BASE}/data/qa_filtered/qa_test.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_pairs.append(json.loads(line.strip()))

cfg_d_preds = []
with open(f'{RESULTS_DIR}/config_D_results.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cfg_d_preds.append(json.loads(line.strip()))

ppo_model.eval()
sample_qs = test_pairs[:5]

print('=== PPO Model vs Config D (Fine-tuned + RAG) ===')
for i, pair in enumerate(sample_qs):
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{pair["question"]}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = ppo_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(ppo_model.pretrained_model.device)
    with torch.no_grad():
        output = ppo_model.pretrained_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    generated = output[0][inputs['input_ids'].shape[1]:]
    ppo_answer = ppo_tokenizer.decode(generated, skip_special_tokens=True).strip()

    print(f'\nQ{i+1}: {pair["question"]}')
    print(f'[Config D]: {cfg_d_preds[i]["prediction"][:200]}...' if len(cfg_d_preds[i]['prediction']) > 200 else f'[Config D]: {cfg_d_preds[i]["prediction"]}')
    print(f'[PPO    ]: {ppo_answer[:200]}...' if len(ppo_answer) > 200 else f'[PPO    ]: {ppo_answer}')

=== PPO Model vs Config D (Fine-tuned + RAG) ===

Q1: Em nghe nói không được ghi âm, chụp ảnh trong giờ học. Có đúng không ạ?
[Config D]: Theo quy chế, sinh viên không được ghi âm, chụp ảnh hoặc sử dụng bất kỳ phương tiện nào để ghi lại nội dung giảng dạy trong giờ học. Đây là một yêu cầu nghiêm ngặt nhằm đảm bảo sinh viên tập trung vào...
[PPO    ]: Đúng rồi, bạn hiểu đúng. Trong giờ học, việc ghi âm hoặc chụp ảnh trên thiết bị cá nhân là không được phép. Điều này nhằm đảm bảo tính minh bạch và tuân thủ quy định về quyền riêng tư cũng như tuân th...

Q2: Em thấy hướng dẫn có nói tránh vừa đi vừa dùng điện thoại. Có bắt buộc không hay chỉ là khuyến cáo ạ?
[Config D]: không tiếp nhận thông tin từ người khác khi chưa được sự đồng ý; (10) không sử dụng thông tin của người khác khi chưa được sự đồng ý; (11) không đưa người ngoài vào trong Trường với mục đích xấu gây ả...
[PPO    ]: Hướng dẫn này thường là một khuyến cáo để đảm bảo an toàn cho người sử dụng. Nó nhắc nhở mọi người nên tập tr

---
**Next step:** Run `07_gradio_demo.ipynb` to build the interactive UI.